In this model, I will add: \
\
Averages after steady state \
\
We're given n as a function of t in the instructions. Based on previous experiments that also have exponential behaviour, we know that any exp^(-t/tau) has a characteristic relaxation time - after time tau, the system returns to an equilibrium. \
\
Given that:
$$ n(t) = \frac{N}{2} [ 1 + e^{\frac{-2t}{N}} ] $$

The relaxation time is proportional to N. We can interpret tau = N / 2 and therefore t_relaxation = N/2 gives:

$$ n(t) = \frac{N}{2} [ 1 + e^{\frac{-2(N)}{N(2)}} ] $$
$$ n(t) = \frac{N}{2} [ 1 + e^{\frac{-2(N)}{N(2)}} ] $$

At t = tau, n(t) = (N/2) (1 + e^{-1}) = (N/2)(1+0.368) = N(0.684) \
\
Which means I will have 68.4% of the particles still on the left box. \
\
Clearly as t -> infinity, n(t) = N/2 \
\
And at t = 0, n(t) = N, which we can see is the case. \
\
As seen below, choosing N, 2N, 3N as time to equilibrium yields pretty much the same. I'll stick to 3N and choose time_steps = 10*N

In [7]:
import numpy as np
import random
import scipy.special
from scipy.special import factorial
import matplotlib.pyplot as plt


# The below commands make the font and image size bigger
plt.rcParams.update({'font.size': 18})
plt.rcParams["figure.figsize"] = (16,9)

In [8]:
k_b = 1.380649e-23                                        # Boltzmann constant for entropy calculation

In [38]:
def ehrenfest_model(N, t_max, t_step):
    pstate = np.zeros(N)                                        # state vector
    time_points = np.arange(0, t_max, t_step)                   # x-axis for plotting
    left_points = []                                            # empty array for left y-values
    right_points = []                                           # empty array for right y-values
    entropy = []

    for i in range(t_max):
    
        n = np.count_nonzero(pstate == 0)                       # keeps track of # particles on the left
        n_right = np.count_nonzero(pstate)                      # keeps track of # particles on the right
        e_coeff =  scipy.special.binom(N, n)
    
        left_points.append(n)
        right_points.append(n_right)
        entropy.append(e_coeff)
    
        rnd = np.random.choice(N)                               # selects a random particle

        if pstate[rnd] == 0:                                    # if particle is on the left, calculates transition prob to right
            if np.random.random() < (n / N):                    # then we generate a random number - if it's smaller than my transition probability
                pstate[rnd] = 1                                 # then the particles moves to the right
        else:
            if np.random.random() < (1 - (n / N)):              # if the particle is on the right, it's a different probability
                pstate[rnd] = 0                                 # and the state changes to left if transition probability > random number

    return n, n_right, time_points, left_points, right_points, N, entropy

https://www.sfu.ca/~jtmulhol/py4math/numpy/02-indexing/

In [96]:
# Stats for N=64
n, n_right, time_points, left_points, right_points, N, entropy = ehrenfest_model(64, 64*10, 1)
steady_state = np.array(left_points[3*N:])

#print(steady_state)
print("The average number of particles on the left in steady-state is:", np.mean(steady_state))
print("The variance of the particles on the left in the steady-state is:", np.var(steady_state))
print("The standard deviation of the particles on the left in steady-state is:", np.std(steady_state))
fluctuation_magnitude = np.std(steady_state) / np.mean(steady_state)
print("The relative magnitude of fluctuations is:", fluctuation_magnitude)

print("Final count of particles on the left:", n)
print("Final count of particles on the right:", n_right)
# print("The standard deviation of the fluctuations is:", np.sqrt(N)/2) -> this is just wrong! 
# this only applies for a binomial distribution when the probability of success is 1/2

# What is the max deviation?
deviations = np.abs(steady_state - N/2)
max_deviation = np.max(deviations)
print(deviations)
print(max_deviation)

The average number of particles on the left in steady-state is: 30.493303571428573
The variance of the particles on the left in the steady-state is: 5.013348014987244
The standard deviation of the particles on the left in steady-state is: 2.2390506950462834
The relative magnitude of fluctuations is: 0.0734276196018399
Final count of particles on the left: 27
Final count of particles on the right: 37
[1. 1. 2. 2. 1. 2. 1. 2. 3. 2. 2. 2. 2. 3. 3. 3. 2. 1. 2. 2. 2. 2. 3. 4.
 3. 2. 2. 1. 1. 1. 0. 0. 0. 1. 2. 2. 2. 3. 4. 4. 4. 4. 4. 4. 5. 5. 5. 4.
 4. 5. 5. 5. 5. 6. 5. 5. 5. 5. 4. 4. 3. 3. 2. 2. 3. 2. 3. 3. 3. 3. 2. 3.
 2. 3. 3. 4. 4. 3. 3. 3. 3. 4. 4. 3. 4. 3. 2. 2. 1. 1. 1. 0. 1. 0. 0. 1.
 2. 1. 0. 1. 2. 2. 3. 4. 5. 4. 4. 3. 2. 1. 1. 1. 1. 0. 1. 1. 2. 3. 3. 3.
 3. 3. 3. 2. 1. 1. 0. 1. 1. 0. 1. 1. 2. 2. 1. 0. 0. 1. 2. 3. 3. 3. 2. 2.
 1. 0. 0. 0. 0. 0. 1. 1. 1. 1. 2. 1. 2. 3. 3. 3. 3. 2. 3. 2. 2. 3. 3. 4.
 3. 3. 3. 3. 4. 4. 3. 3. 2. 2. 2. 1. 0. 0. 1. 0. 1. 2. 2. 1. 0. 1. 1. 2.
 3. 4. 3. 3. 

In [ ]:
# Plotting
plt.plot(time_points, left_points)
plt.plot(time_points, right_points)
plt.plot(time_points, (N / 2) * (1 + np.exp((-2*time_points) / N)))
plt.grid(True)
plt.legend(['Left particles', 'Right Particles', '$n(t) = (N/2)[ 1 + e^{-2t/N}]$'])

In [53]:
# t_eqm = N
# t_eqm_1 = 2*N
# t_eqm_2 = 3*N

# steady_state = np.array(left_points[t_eqm:])
# steady_state_1 = np.array(left_points[t_eqm_1:])
# steady_state_2 = np.array(left_points[t_eqm_2:])

# print(np.mean(steady_state))
# print(np.mean(steady_state_1))
# print(np.mean(steady_state_2))

# print(np.var(steady_state))
# print(np.var(steady_state_1))
# print(np.var(steady_state_2))

# print(np.std(steady_state))
# print(np.std(steady_state_1))
# print(np.std(steady_state_2))

# # It would seem to me that anything after N is pretty good.

In [51]:
print(pstate)
print("Final count of particles on the left:", n)
print("Final count of particles on the right:", n_right)
print("The standard deviation of the fluctuations is:", np.sqrt(N)/2)

# Plotting
plt.plot(time_points, left_points)
plt.plot(time_points, right_points)
plt.plot(time_points, (N / 2) * (1 + np.exp((-2*time_points) / N)))
plt.grid(True)
plt.legend(['Left particles', 'Right Particles', '$n(t) = (N/2)[ 1 + e^{-2t/N}]$'])

In [52]:
# plt.plot(time_points, (k_b * np.log(entropy)))
# plt.grid(True)